In [1]:
# Import packages
import numpy as np
import pandas as pd

# Round the float values in the dataframe to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

In [2]:
"""
Script containing the a-fine aggregation algorithm.

Copyright (c) 2024 Royal Boskalis
"""

from typing import Union


def a_fine_aggregator(
    w: Union[list[float], np.ndarray[float]],
    p: Union[list[list[Union[float, int]]], np.ndarray[list[Union[float, int]]]],
    scores_range: tuple[float, float] = (0.0, 100.0),
) -> np.ndarray[float]:
    """
    Function for aggregating scores in affine spaces, by means of the least square
    distance minimization.

    :param w: weights of the different objectives
    :param p: 2d-array with the scores of the objectives. n-by-m, where n is the number
        of objectives and m the population size
    :return: ndarray with the aggregated scores
    """
    assert len(w) == len(p), (
        f"The number of weights ({len(w)}) is not equal to the number of objectives "
        f"({len(p)})."
    )
    assert (
        round(sum(w), 4) == 1
    ), f"The sum of the weights ({round(sum(w), 4)}) is not equal to 1."

    # transpose the array to make further calculations easier
    p_transposed = np.array(p).transpose()

    # calculate the standard deviation per criteria. If std == 0, a value << 1 is
    # inserted to prevent divide by zero error
    std = np.std(p_transposed, axis=0)
    std[std == 0] = 1e-6

    # calculate the z-score normalized scores per criteria
    z = (p_transposed - np.mean(p_transposed, axis=0)) / std

    # calculate representative preference scores (P_i^*)
    p_star = np.sum(w * z, axis=1)
     
    if len(np.unique(p_star, axis=0)) == 1:
        # if there is only one unique member in p_star
        return np.full(len(p_transposed), -50.0, dtype=float)
    else:
        # return min-max normalized results, so everything is on the scale [0-100]
        # lower and upper bounds for final scaling
        a, b = scores_range
        return -1 * (a + (p_star - min(p_star)) / (max(p_star) - min(p_star)) * (b - a))


if __name__ == '__main__':
    w = [0.5, 0.5]
    #   [obj1_alt1, obj1_alt2, obj1_alt3], [obj2_alt1, obj2_alt2, obj2_alt3]
    p = [[0, 100, 90], [20, 50, 100]]
    scores = a_fine_aggregator(w, p, scores_range=(0.0, -100.0))
    print(scores)


[ -0.          70.78784028 100.        ]


In [12]:
ratings = pd.read_csv(
    "MCDA_input/MCDA_Ratings_3RS.csv",
    sep=";",       # fields are semicolon-separated, not comma-separated
    skiprows=1     # skip the "MCDA Gold Coast Design Alternatives" title row
)

alternatives = list(ratings.columns[2:-1].unique()) # Get the list of alternatives from the dataframe columns, excluding the first two and last column
stakeholders = list(ratings["Stakeholder"].unique()) # Get the list of stakeholders from the "Stakeholder" column in the dataframe
print(f"Alternatives: {alternatives}")
print(f"Stakeholders: {stakeholders}")
display(ratings)


Alternatives: ['Status quo', 'Third runway', 'Regional hub distribution', 'Airport optimalization']
Stakeholders: ['Hong Kong SAR Government', 'Environmental Protection', 'Contractors', 'Local communities']


,Stakeholder,Criteria,Status quo,Third runway,Regional hub distribution,Airport optimalization,Criteria_Weight
0,Hong Kong SAR Government,Economic benefits,35,95,30,70,0.50
1,Hong Kong SAR Government,Passenger safaty,50,90,80,85,0.20
2,Hong Kong SAR Government,International hub position,50,95,55,80,0.30
3,Environmental Protection,Marine ecology impact,100,10,70,60,0.35
4,Environmental Protection,Ecological habitat loss,100,10,40,90,0.25
5,Environmental Protection,Air/ water quality,85,35,75,80,0.25
6,Environmental Protection,Noise pollution,85,35,75,80,0.15
7,Contractors,Technical feasibility,95,55,60,85,0.40
8,Contractors,Construction risk,95,35,75,80,0.35
9,Contractors,Labour and material requirements,95,50,60,80,0.25


In [13]:
# Check each stakeholder's weights sum to 1 (i.e. 100%)
print("Check weights per stakeholder:")
all_valid = True

for stakeholder in stakeholders:
    stakeholder_weights = ratings.loc[ratings["Stakeholder"] == stakeholder, "Criteria_Weight"]
    total = stakeholder_weights.sum()
    is_valid = np.isclose(total, 1)
    all_valid &= is_valid

    status = "OK" if is_valid else "MISMATCH"
    print(f"  {stakeholder:<25s}: {total:6.2f}  [{status}]")


Check weights per stakeholder:
  Hong Kong SAR Government :   1.00  [OK]
  Environmental Protection :   1.00  [OK]
  Contractors              :   1.00  [OK]
  Local communities        :   1.00  [OK]


In [15]:
# Set stakeholder weights
#               Gov, env, con, loc
weights_eq =    [0.25, 0.25, 0.25, 0.25]  # equal weights for stakeholders
weights_dom =   [0.4, 0.3, 0.1, 0.2]

stakeholder_weights = weights_eq 

assert np.isclose(sum(stakeholder_weights), 1), f"Weights must sum to 1, got {sum(stakeholder_weights)}"



In [16]:
# Calculate the aggregated scores for each alternative using the a-fine-aggregator
# --- Level 1: aggregate criteria ratings -> one score per stakeholder per alternative ---
stakeholder_scores = {}

for stakeholder in stakeholders:
    stakeholder_data = ratings.loc[ratings["Stakeholder"] == stakeholder]
    criteria_weights = stakeholder_data["Criteria_Weight"].to_numpy()
    p = stakeholder_data[alternatives].to_numpy()  # shape: n_criteria x n_alternatives
    stakeholder_scores[stakeholder] = a_fine_aggregator(criteria_weights, p, scores_range=(-0.0, -100.0))

# Collect into matrix: rows = stakeholders, columns = alternatives (order matches `alternatives`)
stakeholder_score_matrix = np.array([stakeholder_scores[s] for s in stakeholders])

print("Individual stakeholder aggregated scores:")
display(pd.DataFrame(stakeholder_score_matrix, index=stakeholders, columns=alternatives))

# --- Level 2: aggregate stakeholder scores -> final preference score per alternative ---
final_scores = a_fine_aggregator(stakeholder_weights, stakeholder_score_matrix, scores_range=(-0.0, -100.0))

results = (
    pd.DataFrame(final_scores, index=alternatives, columns=["Preference score"])
    .round(2)
    .sort_values("Preference score", ascending=False)
)

print("Final aggregated preference scores per alternative:")
display(results)


Individual stakeholder aggregated scores:


,Status quo,Third runway,Regional hub distribution,Airport optimalization
Hong Kong SAR Government,0.00,100.00,15.69,67.21
Environmental Protection,100.00,0.00,63.99,76.83
Contractors,100.00,0.00,35.08,72.89
Local communities,100.00,0.00,91.05,94.34


Final aggregated preference scores per alternative:


,Preference score
Airport optimalization,100.00
Status quo,96.30
Regional hub distribution,50.33
Third runway,0.00
